In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import numpy as np
import torch.nn as nn
import pandas as pd

from utils.reproducibility_utils import seed_everything
from utils.data_loading import load_data, load_test_data

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

EXPECTED_ROWS = 49328

Device: cuda


In [ ]:
# EXACT winning architecture - DO NOT CHANGE
class RNN(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.rnn = nn.RNN(input_size, 64, 1, nonlinearity='relu', batch_first=True)
        self.fc = nn.Linear(64, 64)
        self.out = nn.Linear(64, 1)
        self.act = nn.ReLU()
    def forward(self, x):
        x, _ = self.rnn(x)
        return self.out(self.act(self.fc(x[:, -1, :])))

class WeightedL1(nn.Module):
    def __init__(self, threshold=2.0, slope=1.0):
        super().__init__()
        self.threshold = threshold
        self.slope = slope
    def forward(self, pred, target):
        weights = 1.0 + self.slope * torch.clamp(torch.abs(target) - self.threshold, min=0)
        return (torch.abs(pred - target) * weights).mean()

def predict(model, loader):
    model.eval()
    preds = []
    with torch.no_grad():
        for x, _ in loader:
            preds.append(model(x.to(device)).cpu().numpy())
    return np.concatenate(preds).flatten()

def save_submission(preds, filename):
    assert len(preds) == EXPECTED_ROWS, f"Expected {EXPECTED_ROWS}, got {len(preds)}"
    df = pd.DataFrame({"predicted_sideslip": preds, "id": range(len(preds))})
    df.to_csv(filename, index=False)
    print(f"Saved: {filename} | mean={preds.mean():.4f}, std={preds.std():.4f}")

print("Components ready")

✅ Components ready


In [ ]:
# Load data with EXACT same settings as winner
print("Loading data...")
train_loader, val_loader, pipeline = load_data(
    train_path="../datasets_challenge/train",
    train_ratio=0.98, batchsize=64,
    columns_to_drop=["sideSlip", "vyCG", "time", "longitude", "latitude"],
    sequence_length=20, threshold=True, threshold_value=3.0,
    threshold_column="vxCG", target="sideSlip", scaling=True, seed=42,
)
test_loader = load_test_data("../datasets_challenge/test", 512, pipeline, load_anonymized_data=True)
input_size = next(iter(train_loader))[0].shape[2]

test_count = sum(x.shape[0] for x, _ in test_loader)
print(f"Data loaded | Input: {input_size} | Test: {test_count}")
assert test_count == EXPECTED_ROWS

Loading data...
../datasets_challenge/train/train_part_001.csv
../datasets_challenge/train/train_part_002.csv
../datasets_challenge/train/train_part_003.csv
../datasets_challenge/train/train_part_004.csv
../datasets_challenge/train/train_part_005.csv
../datasets_challenge/train/train_part_006.csv
../datasets_challenge/train/train_part_007.csv
../datasets_challenge/train/train_part_008.csv
../datasets_challenge/train/train_part_009.csv
../datasets_challenge/train/train_part_010.csv
../datasets_challenge/train/train_part_011.csv
../datasets_challenge/train/train_part_012.csv
../datasets_challenge/train/train_part_013.csv
../datasets_challenge/train/train_part_014.csv
../datasets_challenge/train/train_part_015.csv
../datasets_challenge/train/train_part_016.csv
../datasets_challenge/train/train_part_017.csv
../datasets_challenge/train/train_part_018.csv
../datasets_challenge/train/train_part_019.csv
../datasets_challenge/train/train_part_020.csv
../datasets_challenge/train/train_part_021.c

In [ ]:
# Training functions with configurable parameters
def train_two_stage(seed, s1_epochs=30, s2_epochs=40, s2_lr=0.00005, threshold=2.0, slope=1.0):
    seed_everything(seed)
    model = RNN(input_size).to(device)
    
    # Stage 1: L1, lr=0.0003
    opt = torch.optim.Adam(model.parameters(), lr=0.0003)
    crit = nn.L1Loss()
    best, state = np.inf, None
    for _ in range(s1_epochs):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(crit(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in val_loader) / len(val_loader)
        if val < best: best, state = val, model.state_dict().copy()
    model.load_state_dict(state)
    
    # Stage 2: WeightedL1
    opt = torch.optim.Adam(model.parameters(), lr=s2_lr)
    crit2 = WeightedL1(threshold=threshold, slope=slope)
    best2, state2 = np.inf, None
    for _ in range(s2_epochs):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit2(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(nn.L1Loss()(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in val_loader) / len(val_loader)
        if val < best2: best2, state2 = val, model.state_dict().copy()
    model.load_state_dict(state2)
    return model

def train_baseline(seed):
    seed_everything(seed)
    model = RNN(input_size).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=0.0003)
    crit = nn.L1Loss()
    best, state, patience = np.inf, None, 0
    for _ in range(50):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(crit(model(x.to(device)), y.to(device).reshape(-1,1)).item() for x,y in val_loader) / len(val_loader)
        if val < best: best, state, patience = val, model.state_dict().copy(), 0
        else:
            patience += 1
            if patience > 10: break
    model.load_state_dict(state)
    return model

print("Training functions ready")

✅ Training functions ready


In [ ]:
# Train baseline models first (needed for all experiments)
SEEDS = [42, 2024]

print("="*60)
print("TRAINING BASELINE MODELS")
print("="*60)

baseline_preds = [predict(train_baseline(s), test_loader) for s in SEEDS]
baseline_ens = np.mean(baseline_preds, axis=0)
print(f"\nBaseline ready | mean={baseline_ens.mean():.4f}")

📦 TRAINING BASELINE MODELS

✅ Baseline ready | mean=-0.0106


---
## Experiment 1: Reproduce Winner (verify setup)

In [ ]:
print("="*60)
print("REPRODUCING WINNER (s2_epochs=40, s2_lr=5e-5)")
print("="*60)

winner_preds = [predict(train_two_stage(s, s2_epochs=40, s2_lr=0.00005), test_loader) for s in SEEDS]
winner_ens = np.mean(winner_preds, axis=0)
winner_blend = (baseline_ens + winner_ens) / 2

save_submission(winner_blend, "pred_v15_winner_40ep.csv")
print("\nExpected: 0.21751 on Kaggle")

🏆 REPRODUCING WINNER (s2_epochs=40, s2_lr=5e-5)
✅ pred_v15_winner_40ep.csv | mean=-0.0017, std=1.4867

📋 Expected: 0.21751 on Kaggle


---
## Experiment 2: Epoch Micro-Tuning (39, 41 epochs)

In [ ]:
print("="*60)
print("EPOCH MICRO-TUNING (39, 41 epochs)")
print("="*60)

for epochs in [39, 41]:
    print(f"\n{epochs} epochs:")
    preds = [predict(train_two_stage(s, s2_epochs=epochs), test_loader) for s in SEEDS]
    ens = np.mean(preds, axis=0)
    blend = (baseline_ens + ens) / 2
    save_submission(blend, f"pred_v15_{epochs}ep.csv")

📦 EPOCH MICRO-TUNING (39, 41 epochs)

📦 39 epochs:
✅ pred_v15_39ep.csv | mean=0.0008, std=1.4901

📦 41 epochs:
✅ pred_v15_41ep.csv | mean=0.0004, std=1.4940


---
## Experiment 3: LR Micro-Tuning (4e-5, 6e-5)

In [ ]:
print("="*60)
print("LR MICRO-TUNING (4e-5, 6e-5)")
print("="*60)

for lr in [0.00004, 0.00006]:
    lr_name = f"{int(lr*100000)}e5"
    print(f"\nlr={lr}:")
    preds = [predict(train_two_stage(s, s2_lr=lr), test_loader) for s in SEEDS]
    ens = np.mean(preds, axis=0)
    blend = (baseline_ens + ens) / 2
    save_submission(blend, f"pred_v15_lr{lr_name}.csv")

📦 LR MICRO-TUNING (4e-5, 6e-5)

📦 lr=4e-05:
✅ pred_v15_lr4e5.csv | mean=-0.0024, std=1.4956

📦 lr=6e-05:
✅ pred_v15_lr6e5.csv | mean=-0.0004, std=1.4961


---
## Experiment 4: Blend Micro-Tuning (48/52, 52/48)

In [ ]:
print("="*60)
print("BLEND MICRO-TUNING")
print("="*60)

# Use winner predictions
for base_w, ts_w in [(0.48, 0.52), (0.52, 0.48)]:
    print(f"\nBaseline {int(base_w*100)}% / Two-stage {int(ts_w*100)}%:")
    blend = baseline_ens * base_w + winner_ens * ts_w
    save_submission(blend, f"pred_v15_b{int(base_w*100)}_ts{int(ts_w*100)}.csv")

📦 BLEND MICRO-TUNING

📦 Baseline 48% / Two-stage 52%:
✅ pred_v15_b48_ts52.csv | mean=-0.0013, std=1.4858

📦 Baseline 52% / Two-stage 48%:
✅ pred_v15_b52_ts48.csv | mean=-0.0020, std=1.4877


---
## Experiment 5: Stage 1 Epochs (28, 32)

In [ ]:
print("="*60)
print("STAGE 1 EPOCH VARIATION (28, 32)")
print("="*60)

for s1_ep in [28, 32]:
    print(f"\nS1={s1_ep} epochs:")
    preds = [predict(train_two_stage(s, s1_epochs=s1_ep, s2_epochs=40), test_loader) for s in SEEDS]
    ens = np.mean(preds, axis=0)
    blend = (baseline_ens + ens) / 2
    save_submission(blend, f"pred_v15_s1_{s1_ep}ep.csv")

📦 STAGE 1 EPOCH VARIATION (28, 32)

📦 S1=28 epochs:
✅ pred_v15_s1_28ep.csv | mean=0.0008, std=1.4970

📦 S1=32 epochs:
✅ pred_v15_s1_32ep.csv | mean=0.0046, std=1.4993


---
## Experiment 6: Weight Threshold (1.8, 2.2)

In [ ]:
print("="*60)
print("WEIGHT THRESHOLD VARIATION (1.8, 2.2)")
print("="*60)

for thresh in [1.8, 2.2]:
    print(f"\nthreshold={thresh}:")
    preds = [predict(train_two_stage(s, threshold=thresh), test_loader) for s in SEEDS]
    ens = np.mean(preds, axis=0)
    blend = (baseline_ens + ens) / 2
    save_submission(blend, f"pred_v15_thresh{str(thresh).replace('.','')}.csv")

📦 WEIGHT THRESHOLD VARIATION (1.8, 2.2)

📦 threshold=1.8:
✅ pred_v15_thresh18.csv | mean=-0.0026, std=1.4867

📦 threshold=2.2:
✅ pred_v15_thresh22.csv | mean=0.0023, std=1.4937


---
## Summary

In [ ]:
import glob

print("="*60)
print("GENERATED FILES")
print("="*60)

files = sorted(glob.glob("pred_v15_*.csv"))
for f in files:
    df = pd.read_csv(f)
    print(f"{f}: {len(df)} rows")

print(f"\nTotal: {len(files)} files")
print("\nCurrent best: 0.21751")
print("\nPriority submissions:")
print("  1. pred_v15_winner_40ep.csv - verify reproduction")
print("  2. pred_v15_39ep.csv - slightly less training")
print("  3. pred_v15_41ep.csv - slightly more training")
print("  4. pred_v15_lr4e5.csv - slightly lower LR")

📋 GENERATED FILES
pred_v15_39ep.csv: 49328 rows
pred_v15_41ep.csv: 49328 rows
pred_v15_b48_ts52.csv: 49328 rows
pred_v15_b52_ts48.csv: 49328 rows
pred_v15_lr4e5.csv: 49328 rows
pred_v15_lr6e5.csv: 49328 rows
pred_v15_s1_28ep.csv: 49328 rows
pred_v15_s1_32ep.csv: 49328 rows
pred_v15_thresh18.csv: 49328 rows
pred_v15_thresh22.csv: 49328 rows
pred_v15_winner_40ep.csv: 49328 rows

🎯 Total: 11 files

🏆 Current best: 0.21751

💡 Priority submissions:
  1. pred_v15_winner_40ep.csv - verify reproduction
  2. pred_v15_39ep.csv - slightly less training
  3. pred_v15_41ep.csv - slightly more training
  4. pred_v15_lr4e5.csv - slightly lower LR
